In [ ]:
# Librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Carga manual del archivo CSV
from google.colab import files
uploaded = files.upload()  # Selecciona archivo CSV con columnas: timestamp, demand

# Leer el archivo
df = pd.read_csv(next(iter(uploaded)))
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
df.set_index('timestamp', inplace=True)

# Normalización [0, 1]
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[['demand']])

# 🪟 Crear secuencias de 24 horas
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 24
X, y = create_sequences(data_scaled, window_size)

# División en entrenamiento, validación y prueba (70/15/15)
train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

# Modelo LSTM
model = Sequential()
model.add(LSTM(64, input_shape=(window_size, 1), return_sequences=False,
               kernel_regularizer=l2(0.001)))  # Regularización L2
model.add(Dropout(0.2))  # Dropout para evitar overfitting
model.add(Dense(1))  # Capa de salida para predicción

# Compilación del modelo
model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Entrenamiento
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,
                    batch_size=32,
                    callbacks=[early_stop],
                    verbose=1)

# Evaluación en conjunto de prueba
y_pred = model.predict(X_test)

# Inversa de la normalización
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred)

# Métricas
mse = mean_squared_error(y_test_inv, y_pred_inv)
mae = mean_absolute_error(y_test_inv, y_pred_inv)
r2 = r2_score(y_test_inv, y_pred_inv)

print("Evaluación del modelo:")
print(f"MSE: {mse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R² Score: {r2:.2f}")

# Gráfico de predicción vs realidad
plt.figure(figsize=(12, 5))
plt.plot(y_test_inv, label='Demanda real', linewidth=2)
plt.plot(y_pred_inv, label='Predicción LSTM', linewidth=2)
plt.title('Predicción vs Realidad - Demanda Energética')
plt.xlabel('Horas')
plt.ylabel('Demanda')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()
